# Simulation

In [ ]:
# Economic Scenario Simulation Model for Upskilling vs Reskilling Programs
# ========================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

# Set visualization styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

class EconomicSimulationModel:
    """
    A dynamic model for simulating economic outcomes of upskilling vs reskilling programs 
    under various economic and investment scenarios.
    """
    
    def __init__(self):
        """Initialize the model with default parameters"""
        # Base program characteristics (derived from your cluster analysis)
        self.base_params = {
            # Upskilling program characteristics
            'upskilling': {
                'cost_per_person': 2500,          # Average cost per person ($)
                'duration_months': 4,             # Average duration in months
                'hours_training': 80,             # Average hours of training
                'coverage_percent': 12,           # % of workforce covered annually
                'completion_rate': 0.85,          # Completion rate
                'base_effectiveness': 0.70,       # Base effectiveness 
                'retention_rate': 0.82,           # Employee retention after program
                'salary_premium': 0.08,           # Average salary increase after program
                'productivity_gain': 0.12,        # Initial productivity gain
                'management_target': 0.35,        # Proportion targeting management levels
                'baseline_roi': 0.25,             # Baseline ROI in normal economic conditions
            },
            # Reskilling program characteristics  
            'reskilling': {
                'cost_per_person': 6000,          # Average cost per person ($)
                'duration_months': 9,             # Average duration in months
                'hours_training': 200,            # Average hours of training
                'coverage_percent': 8,            # % of workforce covered annually
                'completion_rate': 0.75,          # Completion rate
                'base_effectiveness': 0.65,       # Base effectiveness
                'retention_rate': 0.70,           # Employee retention after program
                'salary_premium': 0.15,           # Average salary increase after program
                'productivity_gain': 0.20,        # Initial productivity gain
                'management_target': 0.55,        # Proportion targeting management levels
                'baseline_roi': 0.35,             # Baseline ROI in normal economic conditions
            }
        }
        
        # Economic scenario modifiers
        self.economic_scenarios = {
            'fast_tech_change': {
                'description': 'Rapid technological change, high disruption, strong demand for new skills',
                'upskilling_modifier': 0.7,       # Upskilling becomes less effective
                'reskilling_modifier': 1.5,       # Reskilling becomes more effective
                'upskilling_roi_modifier': 0.8,   # Lower ROI for upskilling
                'reskilling_roi_modifier': 1.4,   # Higher ROI for reskilling
                'job_displacement_rate': 0.15,    # Annual job displacement rate
                'new_job_creation_rate': 0.18,    # Annual new job creation rate
                'skill_obsolescence_rate': 0.25,  # Annual skill obsolescence rate
                'gdp_growth': 0.035,              # Annual GDP growth
                'labor_productivity_growth': 0.03 # Labor productivity growth
            },
            'slow_tech_change': {
                'description': 'Gradual technological change, lower disruption, steady demand for skills',
                'upskilling_modifier': 1.3,       # Upskilling becomes more effective
                'reskilling_modifier': 0.8,       # Reskilling becomes less effective
                'upskilling_roi_modifier': 1.2,   # Higher ROI for upskilling
                'reskilling_roi_modifier': 0.9,   # Lower ROI for reskilling
                'job_displacement_rate': 0.06,    # Annual job displacement rate
                'new_job_creation_rate': 0.08,    # Annual new job creation rate
                'skill_obsolescence_rate': 0.10,  # Annual skill obsolescence rate
                'gdp_growth': 0.025,              # Annual GDP growth
                'labor_productivity_growth': 0.02 # Labor productivity growth
            }
        }
        
        # Investment level modifiers
        self.investment_levels = {
            'low': {
                'description': 'Minimal investment, basic training',
                'cost_modifier': 0.6,             # Lower cost per person
                'effectiveness_modifier': 0.75,   # Lower effectiveness
                'coverage_modifier': 0.6,         # Lower coverage
                'roi_modifier': 0.7,              # Lower ROI
                'scale_modifier': 0.6,            # Lower scaling ability
                'quality_modifier': 0.7           # Lower quality
            },
            'medium': {
                'description': 'Balanced investment, standard training',
                'cost_modifier': 1.0,             # Standard cost per person
                'effectiveness_modifier': 1.0,    # Standard effectiveness
                'coverage_modifier': 1.0,         # Standard coverage
                'roi_modifier': 1.0,              # Standard ROI
                'scale_modifier': 1.0,            # Standard scaling ability
                'quality_modifier': 1.0           # Standard quality
            },
            'high': {
                'description': 'Significant investment, comprehensive training',
                'cost_modifier': 1.6,             # Higher cost per person
                'effectiveness_modifier': 1.35,   # Higher effectiveness
                'coverage_modifier': 1.4,         # Higher coverage
                'roi_modifier': 1.25,             # Higher ROI
                'scale_modifier': 1.5,            # Higher scaling ability
                'quality_modifier': 1.4           # Higher quality
            }
        }
        
        # Company size categories and base metrics
        self.company_sizes = {
            'small': {
                'employees': 100,
                'avg_salary': 65000,
                'training_budget_percent': 1.8,
                'turnover_rate': 0.20,
                'tech_adoption_rate': 0.7
            },
            'medium': {
                'employees': 500,
                'avg_salary': 75000,
                'training_budget_percent': 2.2,
                'turnover_rate': 0.15,
                'tech_adoption_rate': 0.8
            },
            'large': {
                'employees': 2000,
                'avg_salary': 85000,
                'training_budget_percent': 2.5,
                'turnover_rate': 0.12,
                'tech_adoption_rate': 0.9
            }
        }
        
        # Simulation period in years
        self.simulation_years = 5
        
        # Random seed for reproducibility
        np.random.seed(42)
        
    def set_parameters(self, program_type, economic_scenario, investment_level, company_size, 
                       simulation_years=5, custom_params=None):
        """
        Set parameters for the simulation
        
        Args:
            program_type (str): 'upskilling' or 'reskilling'
            economic_scenario (str): 'fast_tech_change' or 'slow_tech_change'
            investment_level (str): 'low', 'medium', or 'high'
            company_size (str): 'small', 'medium', or 'large'
            simulation_years (int): Number of years to simulate
            custom_params (dict): Optional custom parameters to override defaults
        """
        # Validate inputs
        if program_type not in ['upskilling', 'reskilling']:
            raise ValueError("program_type must be 'upskilling' or 'reskilling'")
        
        if economic_scenario not in self.economic_scenarios:
            raise ValueError(f"economic_scenario must be one of {list(self.economic_scenarios.keys())}")
            
        if investment_level not in self.investment_levels:
            raise ValueError(f"investment_level must be one of {list(self.investment_levels.keys())}")
            
        if company_size not in self.company_sizes:
            raise ValueError(f"company_size must be one of {list(self.company_sizes.keys())}")
        
        # Store configuration
        self.program_type = program_type
        self.economic_scenario = economic_scenario
        self.investment_level = investment_level
        self.company_size = company_size
        self.simulation_years = simulation_years
        
        # Get base parameters for the chosen program type
        self.params = self.base_params[program_type].copy()
        
        # Apply modifiers from economic scenario
        eco_modifiers = self.economic_scenarios[economic_scenario]
        self.params['base_effectiveness'] *= eco_modifiers[f'{program_type}_modifier']
        self.params['baseline_roi'] *= eco_modifiers[f'{program_type}_roi_modifier']
        
        # Apply modifiers from investment level
        inv_modifiers = self.investment_levels[investment_level]
        self.params['cost_per_person'] *= inv_modifiers['cost_modifier']
        self.params['base_effectiveness'] *= inv_modifiers['effectiveness_modifier']
        self.params['coverage_percent'] *= inv_modifiers['coverage_modifier']
        self.params['baseline_roi'] *= inv_modifiers['roi_modifier']
        
        # Apply company size parameters
        self.company_params = self.company_sizes[company_size].copy()
        
        # Override with custom parameters if provided
        if custom_params:
            for key, value in custom_params.items():
                if key in self.params:
                    self.params[key] = value
                elif key in self.company_params:
                    self.company_params[key] = value
        
        # Calculate derived metrics based on parameters
        self._calculate_derived_metrics()
        
    def _calculate_derived_metrics(self):
        """Calculate derived metrics based on current parameters"""
        # Number of employees in the company
        employees = self.company_params['employees']
        
        # Number of employees covered by the program annually
        self.params['employees_covered'] = int(employees * self.params['coverage_percent'] / 100)
        
        # Total annual program cost
        self.params['annual_program_cost'] = self.params['employees_covered'] * self.params['cost_per_person']
        
        # Company's annual training budget
        annual_salary_cost = employees * self.company_params['avg_salary']
        self.company_params['training_budget'] = annual_salary_cost * self.company_params['training_budget_percent'] / 100
        
        # Calculate if program fits within budget
        self.params['budget_fit'] = self.params['annual_program_cost'] <= self.company_params['training_budget']
    
    def run_simulation(self):
        """
        Run a multi-year simulation of the training program and its economic outcomes
        """
        # Get parameters from the current configuration
        eco_scenario = self.economic_scenarios[self.economic_scenario]
        
        # Initialize tracking variables
        years = range(self.simulation_years + 1)  # +1 for initial year (year 0)
        employees = self.company_params['employees']
        avg_salary = self.company_params['avg_salary']
        employees_covered = self.params['employees_covered']
        turnover_rate = self.company_params['turnover_rate']
        program_cost = self.params['annual_program_cost']
        baseline_roi = self.params['baseline_roi']
        
        # Tracking arrays for metrics over time
        total_employees = np.zeros(self.simulation_years + 1)
        employees_trained = np.zeros(self.simulation_years + 1)
        cumulative_trained = np.zeros(self.simulation_years + 1)
        program_costs = np.zeros(self.simulation_years + 1)
        cumulative_costs = np.zeros(self.simulation_years + 1)
        productivity_gains = np.zeros(self.simulation_years + 1)
        annual_benefits = np.zeros(self.simulation_years + 1)
        cumulative_benefits = np.zeros(self.simulation_years + 1)
        roi_values = np.zeros(self.simulation_years + 1)
        retention_rates = np.zeros(self.simulation_years + 1)
        skills_relevance = np.zeros(self.simulation_years + 1)
        effectiveness_scores = np.zeros(self.simulation_years + 1)
        salary_increases = np.zeros(self.simulation_years + 1)
        
        # Set initial values
        total_employees[0] = employees
        employees_trained[0] = 0
        cumulative_trained[0] = 0
        program_costs[0] = 0
        cumulative_costs[0] = 0
        productivity_gains[0] = 0
        annual_benefits[0] = 0
        cumulative_benefits[0] = 0
        roi_values[0] = 0
        retention_rates[0] = self.company_params['turnover_rate']
        skills_relevance[0] = 1.0
        effectiveness_scores[0] = self.params['base_effectiveness']
        salary_increases[0] = 0
        
        # Apply technology scenario factors
        skill_obsolescence_rate = eco_scenario['skill_obsolescence_rate']
        gdp_growth = eco_scenario['gdp_growth']
        job_displacement_rate = eco_scenario['job_displacement_rate']
        
        # Run simulation for each year
        for year in range(1, self.simulation_years + 1):
            # Calculate workforce changes
            natural_turnover = total_employees[year-1] * turnover_rate
            tech_displacement = total_employees[year-1] * job_displacement_rate
            
            # New employees (growth & replacement)
            workforce_growth = total_employees[year-1] * (gdp_growth * 0.5)  # Assume 50% of GDP growth translates to workforce growth
            new_employees = workforce_growth + natural_turnover + tech_displacement
            
            # Total employees for this year
            total_employees[year] = total_employees[year-1] - natural_turnover - tech_displacement + new_employees
            
            # Calculate training impact
            employees_trained[year] = min(employees_covered, int(total_employees[year] * self.params['coverage_percent'] / 100))
            cumulative_trained[year] = cumulative_trained[year-1] + employees_trained[year]
            
            # Program costs
            program_costs[year] = employees_trained[year] * self.params['cost_per_person']
            cumulative_costs[year] = cumulative_costs[year-1] + program_costs[year]
            
            # Skill obsolescence effect
            skills_relevance[year] = skills_relevance[year-1] * (1 - skill_obsolescence_rate)
            
            # Effectiveness over time (decays with skill obsolescence but improves with program maturity)
            program_maturity_factor = min(1.0 + (year-1) * 0.05, 1.3)  # Max 30% improvement due to program maturity
            effectiveness_scores[year] = self.params['base_effectiveness'] * program_maturity_factor * skills_relevance[year]
            
            # Calculate retention improvement
            retention_effect = (1 - turnover_rate) * (1 + self.params['retention_rate'] * 0.1 * effectiveness_scores[year])
            retention_rates[year] = min(1 - (turnover_rate * (1 - self.params['retention_rate'] * 0.1)), 0.95)
            
            # Salary increases for trained employees
            salary_increases[year] = self.params['salary_premium'] * effectiveness_scores[year]
            
            # Productivity gains decay over time but are refreshed by new training
            new_productivity = self.params['productivity_gain'] * effectiveness_scores[year]
            ongoing_productivity = productivity_gains[year-1] * (1 - skill_obsolescence_rate)
            
            # Weighted productivity gain calculation (accounting for new and previously trained employees)
            if cumulative_trained[year-1] > 0:
                productivity_gains[year] = (ongoing_productivity * (cumulative_trained[year-1] - employees_trained[year-1] * turnover_rate) + 
                                          new_productivity * employees_trained[year]) / cumulative_trained[year]
            else:
                productivity_gains[year] = new_productivity
            
            # Calculate annual benefits
            retention_savings = natural_turnover * avg_salary * 0.5 * (retention_rates[year] - (1-turnover_rate))
            productivity_benefit = employees_trained[year] * avg_salary * productivity_gains[year]
            redeployment_savings = 0
            
            # Additional benefits for reskilling programs (avoiding hiring costs)
            if self.program_type == 'reskilling':
                redeployment_savings = tech_displacement * avg_salary * 0.3  # 30% of salary is the cost to hire new employees
            
            annual_benefits[year] = productivity_benefit + retention_savings + redeployment_savings
            cumulative_benefits[year] = cumulative_benefits[year-1] + annual_benefits[year]
            
            # Calculate ROI
            if cumulative_costs[year] > 0:
                roi_values[year] = (cumulative_benefits[year] - cumulative_costs[year]) / cumulative_costs[year]
            else:
                roi_values[year] = 0
        
        # Store simulation results in a DataFrame
        self.results = pd.DataFrame({
            'Year': years,
            'Total_Employees': total_employees,
            'Employees_Trained': employees_trained,
            'Cumulative_Trained': cumulative_trained,
            'Program_Costs': program_costs,
            'Cumulative_Costs': cumulative_costs,
            'Productivity_Gains': productivity_gains,
            'Annual_Benefits': annual_benefits,
            'Cumulative_Benefits': cumulative_benefits,
            'ROI': roi_values,
            'Retention_Rate': retention_rates,
            'Skills_Relevance': skills_relevance,
            'Effectiveness': effectiveness_scores,
            'Salary_Increases': salary_increases
        })
        
        return self.results
        
    def compare_scenarios(self, scenario_configs, metrics=['ROI', 'Cumulative_Benefits', 'Cumulative_Costs']):
        """
        Compare multiple scenarios side by side
        
        Args:
            scenario_configs (list): List of dictionaries with scenario configurations
            metrics (list): List of metrics to compare
        
        Returns:
            dict: Dictionary of DataFrames with comparison results
        """
        comparison_results = {}
        
        # Run each scenario and collect results
        scenario_results = []
        for i, config in enumerate(scenario_configs):
            # Store current configuration
            current_config = {
                'program_type': self.program_type if hasattr(self, 'program_type') else None,
                'economic_scenario': self.economic_scenario if hasattr(self, 'economic_scenario') else None,
                'investment_level': self.investment_level if hasattr(self, 'investment_level') else None,
                'company_size': self.company_size if hasattr(self, 'company_size') else None,
                'simulation_years': self.simulation_years
            }
            
            # Setup and run the new scenario
            self.set_parameters(**config)
            results = self.run_simulation()
            results['Scenario'] = f"Scenario {i+1}: {config['program_type'].capitalize()}, {config['economic_scenario']}, {config['investment_level']} investment"
            scenario_results.append(results)
            
            # Restore original configuration
            if all(current_config.values()):
                self.set_parameters(**current_config)
        
        # Combine results for each metric
        for metric in metrics:
            comparison_data = []
            for i, results in enumerate(scenario_results):
                data = results[['Year', metric, 'Scenario']].copy()
                comparison_data.append(data)
            
            comparison_results[metric] = pd.concat(comparison_data)
        
        return comparison_results
    
    def plot_simulation_results(self, metrics=None, figsize=(15, 10)):
        """
        Plot the simulation results for the specified metrics
        
        Args:
            metrics (list): List of metrics to plot. If None, will plot standard metrics.
            figsize (tuple): Figure size in inches
        """
        if not hasattr(self, 'results'):
            print("No simulation results available. Run simulation first.")
            return
        
        if metrics is None:
            # Default metrics to plot
            metrics = [
                ('ROI', 'Return on Investment'),
                ('Cumulative_Benefits', 'Cumulative Benefits ($)'),
                ('Cumulative_Costs', 'Cumulative Costs ($)'),
                ('Productivity_Gains', 'Productivity Gains (%)'),
                ('Effectiveness', 'Program Effectiveness'),
                ('Skills_Relevance', 'Skills Relevance Over Time')
            ]
        
        # Create subplots
        n_metrics = len(metrics)
        rows = (n_metrics + 1) // 2  # Ceiling division
        fig, axes = plt.subplots(rows, 2, figsize=figsize)
        axes = axes.flatten()
        
        # Program type for title
        program_title = 'Upskilling' if self.program_type == 'upskilling' else 'Reskilling'
        
        # Plot each metric
        for i, (metric, title) in enumerate(metrics):
            if i < len(axes):
                ax = axes[i]
                self.results.plot(x='Year', y=metric, ax=ax, marker='o', linewidth=2)
                ax.set_title(title)
                ax.set_xlabel('Year')
                ax.set_ylabel(title)
                ax.grid(True)
                
                # Add annotations for key values
                for year in [1, self.simulation_years]:
                    value = self.results.loc[self.results['Year'] == year, metric].values[0]
                    if 'Costs' in metric or 'Benefits' in metric:
                        # Format as currency for money values
                        label = f"${value:,.0f}"
                    elif 'ROI' in metric:
                        # Format as percentage for ROI
                        label = f"{value*100:.1f}%"
                    else:
                        # Format as decimal for other metrics
                        label = f"{value:.3f}"
                    
                    ax.annotate(label, 
                               (year, value),
                               textcoords="offset points", 
                               xytext=(0,10), 
                               ha='center')
        
        # Remove any unused subplots
        for i in range(n_metrics, len(axes)):
            fig.delaxes(axes[i])
        
        # Add overall title
        fig.suptitle(f'{program_title} Program Simulation Results\n'
                    f'Economic Scenario: {self.economic_scenario}, '
                    f'Investment Level: {self.investment_level}, '
                    f'Company Size: {self.company_size}', 
                    fontsize=16)
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        return fig, axes
    
    def plot_scenario_comparison(self, comparison_results, figsize=(15, 10)):
        """
        Plot the comparison of multiple scenarios
        
        Args:
            comparison_results (dict): Dictionary of DataFrames with comparison results
            figsize (tuple): Figure size in inches
        """
        metrics = list(comparison_results.keys())
        n_metrics = len(metrics)
        
        # Create subplots
        rows = (n_metrics + 1) // 2  # Ceiling division
        fig, axes = plt.subplots(rows, 2, figsize=figsize)
        axes = axes.flatten()
        
        # Plot each metric
        for i, metric in enumerate(metrics):
            if i < len(axes):
                ax = axes[i]
                data = comparison_results[metric]
                
                # Plot each scenario
                for scenario in data['Scenario'].unique():
                    scenario_data = data[data['Scenario'] == scenario]
                    ax.plot(scenario_data['Year'], scenario_data[metric], marker='o', linewidth=2, label=scenario)
                
                ax.set_title(metric.replace('_', ' '))
                ax.set_xlabel('Year')
                ax.set_ylabel(metric.replace('_', ' '))
                ax.grid(True)
                ax.legend(fontsize=8)
                
                # Format y-axis for money values
                if 'Costs' in metric or 'Benefits' in metric:
                    from matplotlib.ticker import FuncFormatter
                    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x:,.0f}'))
                
                # Format y-axis for percentage values
                if metric == 'ROI':
                    from matplotlib.ticker import FuncFormatter
                    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
        
        # Remove any unused subplots
        for i in range(n_metrics, len(axes)):
            fig.delaxes(axes[i])
        
        fig.suptitle('Comparison of Program Scenarios', fontsize=16)
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        return fig, axes
        
    def generate_program_report(self):
        """Generate a text report of the simulation results"""
        if not hasattr(self, 'results'):
            return "No simulation results available. Run simulation first."
        
        # Get final year results
        final_year = self.results.iloc[-1]
        initial_year = self.results.iloc[0]
        
        # Format the program type for display
        program_type = self.program_type.capitalize()
        
        report = [
            f"# {program_type} Program Economic Simulation Report",
            f"\n## Scenario Parameters",
            f"* Program Type: {program_type}",
            f"* Economic Scenario: {self.economic_scenario.replace('_', ' ').title()}",
            f"* Investment Level: {self.investment_level.title()}",
            f"* Company Size: {self.company_size.title()} ({self.company_params['employees']} employees)",
            f"* Simulation Period: {self.simulation_years} years",
            
            f"\n## Program Design",
            f"* Cost per Person: ${self.params['cost_per_person']:,.0f}",
            f"* Duration: {self.params['duration_months']} months",
            f"* Training Hours: {self.params['hours_training']} hours",
            f"* Annual Coverage: {self.params['coverage_percent']}% of workforce",
            f"* Total Employees Covered: {final_year['Cumulative_Trained']:,.0f} over {self.simulation_years} years",
            
            f"\n## Financial Outcomes",
            f"* Total Program Cost: ${final_year['Cumulative_Costs']:,.0f}",
            f"* Total Benefits Generated: ${final_year['Cumulative_Benefits']:,.0f}",
            f"* Net Value Created: ${final_year['Cumulative_Benefits'] - final_year['Cumulative_Costs']:,.0f}",
            f"* Final ROI: {final_year['ROI']*100:.1f}%",
            f"* Break-even Point: {self._calculate_breakeven_point()} years",
            
            f"\n## Performance Metrics",
            f"* Final Retention Rate: {final_year['Retention_Rate']*100:.1f}% (vs. initial {initial_year['Retention_Rate']*100:.1f}%)",
            f"* Final Productivity Gain: {final_year['Productivity_Gains']*100:.1f}%",
            f"* Skills Relevance after {self.simulation_years} years: {final_year['Skills_Relevance']*100:.1f}%",
            f"* Program Effectiveness: {final_year['Effectiveness']*100:.1f}%",
            
            f"\n## Conclusion",
            self._generate_conclusion()
        ]
        
        return "\n".join(report)
    
    def _calculate_breakeven_point(self):
        """Calculate the breakeven point in years"""
        if not hasattr(self, 'results'):
            return "N/A"
        
        # Find where cumulative benefits exceed cumulative costs
        breakeven = self.results[self.results['Cumulative_Benefits'] >= self.results['Cumulative_Costs']]
        
        if len(breakeven) == 0:
            return f"Beyond {self.simulation_years} years"
        else:
            # Get the first year where benefits exceed costs
            breakeven_year = breakeven['Year'].iloc[0]
            
            # If it's year 0, return 'Immediate'
            if breakeven_year == 0:
                return "Immediate"
            
            # If it's the first year with data, calculate fraction of year
            if breakeven_year == 1:
                # Linear interpolation to estimate fraction of year
                year_0 = self.results[self.results['Year'] == 0].iloc[0]
                year_1 = self.results[self.results['Year'] == 1].iloc[0]
                
                costs_0 = year_0['Cumulative_Costs']
                costs_1 = year_1['Cumulative_Costs']
                benefits_0 = year_0['Cumulative_Benefits']
                benefits_1 = year_1['Cumulative_Benefits']
                
                # If no change in costs or benefits, return breakeven_year
                if costs_1 == costs_0 or benefits_1 == benefits_0:
                    return f"{breakeven_year:.1f}"
                
                # Calculate the fraction of the year
                net_0 = benefits_0 - costs_0
                net_1 = benefits_1 - costs_1
                
                # If both are negative or both are positive, return breakeven_year
                if (net_0 < 0 and net_1 < 0) or (net_0 >= 0 and net_1 >= 0):
                    return f"{breakeven_year:.1f}"
                
                fraction = -net_0 / (net_1 - net_0)
                return f"{breakeven_year - 1 + fraction:.1f}"
            
            return f"{breakeven_year:.1f}"
    
def _generate_conclusion(self):
        """Generate a conclusion based on the simulation results"""
        if not hasattr(self, 'results'):
            return "No simulation results available."
        
        # Get final year results
        final_year = self.results.iloc[-1]
        
        # Determine if the program is financially successful
        if final_year['ROI'] > 0.2:
            financial_success = "highly successful"
        elif final_year['ROI'] > 0:
            financial_success = "moderately successful"
        else:
            financial_success = "not financially viable"
        
        # Analyze effectiveness and skills relevance
        if final_year['Effectiveness'] > 0.7:
            effectiveness = "very effective"
        elif final_year['Effectiveness'] > 0.5:
            effectiveness = "moderately effective"
        else:
            effectiveness = "not sufficiently effective"
            
        if final_year['Skills_Relevance'] > 0.7:
            skills_relevance = "maintaining high relevance"
        elif final_year['Skills_Relevance'] > 0.5:
            skills_relevance = "maintaining moderate relevance"
        else:
            skills_relevance = "facing significant skill obsolescence"
            
        # Generate conclusion text
        program_type = self.program_type.capitalize()
        conclusion = f"The {program_type} program under a {self.economic_scenario.replace('_', ' ')} economic scenario with {self.investment_level} investment is {financial_success} financially, {effectiveness} in achieving its objectives, and {skills_relevance} over the {self.simulation_years}-year period. "
        
        # Add economic scenario specific insights
        if self.economic_scenario == 'fast_tech_change':
            if self.program_type == 'reskilling':
                conclusion += f"In a rapidly changing technological environment, this reskilling program provides essential preparation for workforce transitions, with an ROI of {final_year['ROI']*100:.1f}%. "
                if self.investment_level == 'high':
                    conclusion += "The high investment level is justified by the substantial benefits in employee redeployment and retention."
                else:
                    conclusion += f"A higher investment level might yield even better results given the pace of technological change."
            else: # upskilling
                conclusion += f"While upskilling shows a positive ROI of {final_year['ROI']*100:.1f}%, the rapid pace of technological change suggests that more comprehensive reskilling might be needed for long-term workforce adaptability. "
                conclusion += "Consider complementing with targeted reskilling initiatives for the most disrupted roles."
        else: # slow_tech_change
            if self.program_type == 'upskilling':
                conclusion += f"In a steady technological environment, this upskilling program efficiently builds on existing capabilities with an ROI of {final_year['ROI']*100:.1f}%. "
                if self.investment_level == 'low':
                    conclusion += "Even with modest investment, the program delivers meaningful results by focusing on incremental skill improvements."
                else:
                    conclusion += "The investment yields consistent returns through productivity improvements and employee retention."
            else: # reskilling
                conclusion += f"While the reskilling program shows an ROI of {final_year['ROI']*100:.1f}%, the slower pace of technological change suggests that a more targeted approach or partial shift to upskilling might be more cost-effective. "
                conclusion += "Consider a more balanced portfolio of skill development initiatives."
        
        return conclusion

def create_interactive_dashboard():
    """Create an interactive dashboard for the simulation model"""
    model = EconomicSimulationModel()
    
    # Create widgets for parameters
    program_dropdown = widgets.Dropdown(
        options=[('Upskilling', 'upskilling'), ('Reskilling', 'reskilling')],
        value='upskilling',
        description='Program Type:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    economic_dropdown = widgets.Dropdown(
        options=[('Fast Technological Change', 'fast_tech_change'), 
                ('Slow Technological Change', 'slow_tech_change')],
        value='fast_tech_change',
        description='Economic Scenario:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    investment_dropdown = widgets.Dropdown(
        options=[('Low', 'low'), ('Medium', 'medium'), ('High', 'high')],
        value='medium',
        description='Investment Level:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    company_dropdown = widgets.Dropdown(
        options=[('Small', 'small'), ('Medium', 'medium'), ('Large', 'large')],
        value='medium',
        description='Company Size:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    years_slider = widgets.IntSlider(
        value=5,
        min=2,
        max=10,
        step=1,
        description='Simulation Years:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    # Create tabs for different views
    tab_names = ['Simulation Results', 'Scenario Comparison', 'Report']
    tabs = widgets.Tab()
    tabs.children = [widgets.Output() for _ in range(len(tab_names))]
    for i, name in enumerate(tab_names):
        tabs.set_title(i, name)
    
    # Create run button
    run_button = widgets.Button(
        description='Run Simulation',
        button_style='success',
        tooltip='Click to run the simulation with the current parameters',
        icon='play'
    )
    
    # Create comparison button
    compare_button = widgets.Button(
        description='Run Comparison',
        button_style='info',
        tooltip='Compare multiple scenarios',
        icon='random'
    )
    
    # Create widgets for comparison scenarios
    scenario1_program = widgets.Dropdown(
        options=[('Upskilling', 'upskilling'), ('Reskilling', 'reskilling')],
        value='upskilling',
        description='Scenario 1:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    scenario1_investment = widgets.Dropdown(
        options=[('Low', 'low'), ('Medium', 'medium'), ('High', 'high')],
        value='medium',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    scenario2_program = widgets.Dropdown(
        options=[('Upskilling', 'upskilling'), ('Reskilling', 'reskilling')],
        value='reskilling',
        description='Scenario 2:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    scenario2_investment = widgets.Dropdown(
        options=[('Low', 'low'), ('Medium', 'medium'), ('High', 'high')],
        value='medium',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    # Function to run the simulation
    def run_simulation(b):
        with tabs.children[0]:
            clear_output()
            
            # Get parameters from widgets
            program_type = program_dropdown.value
            economic_scenario = economic_dropdown.value
            investment_level = investment_dropdown.value
            company_size = company_dropdown.value
            simulation_years = years_slider.value
            
            # Set parameters and run simulation
            model.set_parameters(
                program_type=program_type,
                economic_scenario=economic_scenario,
                investment_level=investment_level,
                company_size=company_size,
                simulation_years=simulation_years
            )
            
            results = model.run_simulation()
            
            # Display results
            print(f"Simulation Results for {program_type.capitalize()} Program")
            print(f"Economic Scenario: {economic_scenario}, Investment Level: {investment_level}, Company Size: {company_size}")
            print(f"Simulation Years: {simulation_years}")
            print("-" * 80)
            
            # Display key metrics
            final_year = results.iloc[-1]
            print(f"Final ROI: {final_year['ROI']*100:.2f}%")
            print(f"Total Investment: ${final_year['Cumulative_Costs']:,.2f}")
            print(f"Total Benefits: ${final_year['Cumulative_Benefits']:,.2f}")
            print(f"Net Value Created: ${final_year['Cumulative_Benefits'] - final_year['Cumulative_Costs']:,.2f}")
            print(f"Break-even Point: {model._calculate_breakeven_point()} years")
            print("-" * 80)
            
            # Plot results
            model.plot_simulation_results()
            plt.show()
    
    # Function to run comparison
    def run_comparison(b):
        with tabs.children[1]:
            clear_output()
            
            # Get economic scenario from widgets
            economic_scenario = economic_dropdown.value
            company_size = company_dropdown.value
            simulation_years = years_slider.value
            
            # Create scenario configurations
            scenario_configs = [
                {
                    'program_type': scenario1_program.value,
                    'economic_scenario': economic_scenario,
                    'investment_level': scenario1_investment.value,
                    'company_size': company_size,
                    'simulation_years': simulation_years
                },
                {
                    'program_type': scenario2_program.value,
                    'economic_scenario': economic_scenario,
                    'investment_level': scenario2_investment.value,
                    'company_size': company_size,
                    'simulation_years': simulation_years
                }
            ]
            
            # Run comparison
            comparison_results = model.compare_scenarios(scenario_configs)
            
            # Display results
            print(f"Scenario Comparison Results")
            print(f"Economic Scenario: {economic_scenario}, Company Size: {company_size}")
            print(f"Simulation Years: {simulation_years}")
            print("-" * 80)
            
            # Plot comparison
            model.plot_scenario_comparison(comparison_results)
            plt.show()
            
            # Display summary table
            print("\nSummary Comparison Table (Final Year):")
            summary_data = []
            for i, config in enumerate(scenario_configs):
                # Run simulation for this config
                model.set_parameters(**config)
                results = model.run_simulation()
                final_year = results.iloc[-1]
                
                summary_data.append({
                    'Scenario': f"Scenario {i+1}: {config['program_type'].capitalize()}, {config['investment_level']} investment",
                    'ROI': f"{final_year['ROI']*100:.2f}%",
                    'Total Cost': f"${final_year['Cumulative_Costs']:,.0f}",
                    'Total Benefits': f"${final_year['Cumulative_Benefits']:,.0f}",
                    'Net Value': f"${final_year['Cumulative_Benefits'] - final_year['Cumulative_Costs']:,.0f}",
                    'Breakeven': model._calculate_breakeven_point()
                })
            
            display(pd.DataFrame(summary_data).set_index('Scenario'))
    
    # Function to update report tab
    def update_report_tab(change):
        if tabs.selected_index == 2:
            with tabs.children[2]:
                clear_output()
                
                # Get parameters from widgets
                program_type = program_dropdown.value
                economic_scenario = economic_dropdown.value
                investment_level = investment_dropdown.value
                company_size = company_dropdown.value
                simulation_years = years_slider.value
                
                # Set parameters and run simulation if needed
                if not hasattr(model, 'results'):
                    model.set_parameters(
                        program_type=program_type,
                        economic_scenario=economic_scenario,
                        investment_level=investment_level,
                        company_size=company_size,
                        simulation_years=simulation_years
                    )
                    model.run_simulation()
                
                # Generate and display report
                report = model.generate_program_report()
                display(HTML(f"<pre style='font-family:Arial; font-size:14px; white-space: pre-wrap;'>{report}</pre>"))
    
    # Connect button callbacks
    run_button.on_click(run_simulation)
    compare_button.on_click(run_comparison)
    tabs.observe(update_report_tab, names='selected_index')
    
    # Create layout
    params_box = widgets.VBox([
        widgets.HTML("<h2>Simulation Parameters</h2>"),
        program_dropdown,
        economic_dropdown,
        investment_dropdown,
        company_dropdown,
        years_slider,
        run_button
    ])
    
    comparison_box = widgets.VBox([
        widgets.HTML("<h2>Scenario Comparison</h2>"),
        widgets.HBox([scenario1_program, scenario1_investment]),
        widgets.HBox([scenario2_program, scenario2_investment]),
        compare_button
    ])
    
    # Main dashboard layout
    dashboard = widgets.VBox([
        widgets.HTML("<h1>Economic Scenario Simulation Model for Upskilling vs Reskilling Programs</h1>"),
        widgets.HBox([params_box, comparison_box]),
        tabs
    ])
    
    return dashboard

# Create a Monte Carlo simulation function to handle uncertainty
def monte_carlo_simulation(base_model, num_simulations=100, risk_factors=None):
    """
    Run a Monte Carlo simulation with varying parameters to account for uncertainty
    
    Args:
        base_model: Configured EconomicSimulationModel instance
        num_simulations: Number of simulation runs
        risk_factors: Dictionary of parameters and their standard deviations
        
    Returns:
        DataFrame with simulation results
    """
    if not risk_factors:
        # Default risk factors (parameter, standard deviation as % of mean)
        risk_factors = {
            'cost_per_person': 0.15,          # 15% variation in cost
            'productivity_gain': 0.20,         # 20% variation in productivity gains
            'retention_rate': 0.10,            # 10% variation in retention rate
            'base_effectiveness': 0.15,        # 15% variation in program effectiveness
            'completion_rate': 0.10            # 10% variation in completion rate
        }
    
    # Store results for each simulation
    all_results = []
    
    # Store current parameters to restore later
    original_params = base_model.params.copy()
    
    # Run simulations
    for i in range(num_simulations):
        # Create a copy of original parameters
        new_params = original_params.copy()
        
        # Apply random variations to parameters based on risk factors
        for param, std_pct in risk_factors.items():
            if param in new_params:
                mean_value = new_params[param]
                std_dev = mean_value * std_pct
                # Generate a random value from normal distribution
                random_value = np.random.normal(mean_value, std_dev)
                # Ensure values stay positive and reasonable
                new_params[param] = max(0, random_value)
        
        # Set the new parameters and run simulation
        base_model.params = new_params
        base_model._calculate_derived_metrics()
        results = base_model.run_simulation()
        
        # Add simulation ID and store results
        results['Simulation'] = i + 1
        all_results.append(results)
    
    # Restore original parameters
    base_model.params = original_params
    base_model._calculate_derived_metrics()
    
    # Combine all simulation results
    combined_results = pd.concat(all_results)
    
    return combined_results

def plot_monte_carlo_results(monte_carlo_results, metric='ROI', confidence_interval=0.9):
    """
    Plot Monte Carlo simulation results with confidence intervals
    
    Args:
        monte_carlo_results: DataFrame with Monte Carlo simulation results
        metric: Metric to plot (e.g., 'ROI', 'Cumulative_Benefits')
        confidence_interval: Confidence interval to display (0 to 1)
    """
    # Get unique years and simulations
    years = monte_carlo_results['Year'].unique()
    simulations = monte_carlo_results['Simulation'].unique()
    
    # Create figure
    plt.figure(figsize=(12, 8))
    
    # Calculate statistics for each year
    mean_values = []
    lower_bounds = []
    upper_bounds = []
    
    alpha = (1 - confidence_interval) / 2
    percentiles = [alpha * 100, (1 - alpha) * 100]
    
    for year in years:
        year_data = monte_carlo_results[monte_carlo_results['Year'] == year][metric]
        mean_values.append(year_data.mean())
        lower, upper = np.percentile(year_data, percentiles)
        lower_bounds.append(lower)
        upper_bounds.append(upper)
    
    # Plot mean line
    plt.plot(years, mean_values, 'b-', linewidth=2, label=f'Mean {metric}')
    
    # Plot confidence interval
    plt.fill_between(years, lower_bounds, upper_bounds, color='b', alpha=0.2, 
                     label=f'{confidence_interval*100:.0f}% Confidence Interval')
    
    # Plot individual simulations (semi-transparent)
    if len(simulations) <= 100:  # Only plot if not too many simulations
        for sim in simulations:
            sim_data = monte_carlo_results[monte_carlo_results['Simulation'] == sim]
            plt.plot(sim_data['Year'], sim_data[metric], 'k-', alpha=0.05)
    
    # Format plot
    plt.title(f'Monte Carlo Simulation Results for {metric}')
    plt.xlabel('Year')
    
    # Format y-axis based on metric
    if metric == 'ROI':
        plt.ylabel('Return on Investment')
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2%}'))
    elif 'Cost' in metric or 'Benefit' in metric:
        plt.ylabel(f'{metric} ($)')
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    else:
        plt.ylabel(metric)
    
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    return plt.gcf()

# Example usage of the model
#if __name__ == "__main__":
    # Create and display the interactive dashboard
    #dashboard = create_interactive_dashboard()
    #display(dashboard)

In [ ]:
# Economic Scenario Simulations: Upskilling vs Reskilling Programs
# ===============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

# Assuming the EconomicSimulationModel from the previous code is available
# If running this separately, import the model from the previous file
# from economic_simulation_model import EconomicSimulationModel

# Set visualization styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 14

# Initialize the model
model = EconomicSimulationModel()

def run_scenario_analysis():
    """Run analysis for key scenarios and display results"""
    print("Running scenario analysis for upskilling and reskilling programs...\n")
    
    # Define scenarios to compare
    scenarios = [
        {
            'name': 'Upskilling - Fast Tech Change - Low Investment',
            'config': {
                'program_type': 'upskilling',
                'economic_scenario': 'fast_tech_change',
                'investment_level': 'low',
                'company_size': 'medium',
                'simulation_years': 5
            }
        },
        {
            'name': 'Upskilling - Fast Tech Change - High Investment',
            'config': {
                'program_type': 'upskilling',
                'economic_scenario': 'fast_tech_change',
                'investment_level': 'high',
                'company_size': 'medium',
                'simulation_years': 5
            }
        },
        {
            'name': 'Reskilling - Fast Tech Change - Low Investment',
            'config': {
                'program_type': 'reskilling',
                'economic_scenario': 'fast_tech_change',
                'investment_level': 'low',
                'company_size': 'medium',
                'simulation_years': 5
            }
        },
        {
            'name': 'Reskilling - Fast Tech Change - High Investment',
            'config': {
                'program_type': 'reskilling',
                'economic_scenario': 'fast_tech_change',
                'investment_level': 'high',
                'company_size': 'medium',
                'simulation_years': 5
            }
        },
        {
            'name': 'Upskilling - Slow Tech Change - Medium Investment',
            'config': {
                'program_type': 'upskilling',
                'economic_scenario': 'slow_tech_change',
                'investment_level': 'medium',
                'company_size': 'medium',
                'simulation_years': 5
            }
        },
        {
            'name': 'Reskilling - Slow Tech Change - Medium Investment',
            'config': {
                'program_type': 'reskilling',
                'economic_scenario': 'slow_tech_change',
                'investment_level': 'medium',
                'company_size': 'medium',
                'simulation_years': 5
            }
        }
    ]
    
    # Run simulations for each scenario
    results = []
    for scenario in scenarios:
        print(f"Simulating: {scenario['name']}...")
        model.set_parameters(**scenario['config'])
        sim_results = model.run_simulation()
        # Get final year results
        final_year = sim_results.iloc[-1].copy()
        final_year['Scenario'] = scenario['name']
        final_year['Program'] = scenario['config']['program_type']
        final_year['Economic'] = scenario['config']['economic_scenario']
        final_year['Investment'] = scenario['config']['investment_level']
        results.append(final_year)
    
    # Create summary dataframe
    summary_df = pd.DataFrame(results)
    
    # Create summary table
    summary_table = summary_df[['Scenario', 'ROI', 'Cumulative_Costs', 'Cumulative_Benefits', 
                               'Productivity_Gains', 'Effectiveness']]
    
    # Format the table for display
    formatted_table = summary_table.copy()
    formatted_table['ROI'] = formatted_table['ROI'].apply(lambda x: f"{x*100:.1f}%")
    formatted_table['Cumulative_Costs'] = formatted_table['Cumulative_Costs'].apply(lambda x: f"${x:,.0f}")
    formatted_table['Cumulative_Benefits'] = formatted_table['Cumulative_Benefits'].apply(lambda x: f"${x:,.0f}")
    formatted_table['Productivity_Gains'] = formatted_table['Productivity_Gains'].apply(lambda x: f"{x*100:.1f}%")
    formatted_table['Effectiveness'] = formatted_table['Effectiveness'].apply(lambda x: f"{x*100:.1f}%")
    
    print("\n=== SUMMARY OF SCENARIO RESULTS ===")
    print(formatted_table.to_string(index=False))
    
    return summary_df

def create_comparative_visualizations(summary_df):
    """Create comparative visualizations of scenario results"""
    print("\nGenerating comparative visualizations...")
    
    # 1. ROI Comparison by Program Type and Investment Level
    plt.figure(figsize=(14, 8))
    
    # Group by program type and investment level
    grouped_data = summary_df.groupby(['Program', 'Investment'])
    
    # Create bar positions
    programs = summary_df['Program'].unique()
    investments = summary_df['Investment'].unique()
    
    # Set up the bar chart
    x = np.arange(len(programs))  # positions for program types
    width = 0.2  # width of bars
    
    # Plot bars for each investment level
    for i, inv in enumerate(investments):
        if inv not in ['low', 'medium', 'high']:
            continue
            
        roi_values = []
        for prog in programs:
            group_data = grouped_data.get_group((prog, inv)) if (prog, inv) in grouped_data.groups else None
            if group_data is not None:
                # If multiple rows for this combination, take the first one
                roi = group_data['ROI'].values[0]
                roi_values.append(roi)
            else:
                roi_values.append(0)
        
        label = inv.capitalize()
        plt.bar(x + (i-1)*width, roi_values, width, label=f'{label} Investment')
    
    # Customize the plot
    plt.xlabel('Program Type')
    plt.ylabel('Return on Investment (ROI)')
    plt.title('ROI Comparison by Program Type and Investment Level')
    plt.xticks(x, [p.capitalize() for p in programs])
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    
    # Format y-axis as percentage
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:.0f}%'))
    
    # Add value labels on bars
    for i, inv in enumerate(investments):
        if inv not in ['low', 'medium', 'high']:
            continue
            
        for j, prog in enumerate(programs):
            group_data = grouped_data.get_group((prog, inv)) if (prog, inv) in grouped_data.groups else None
            if group_data is not None:
                roi = group_data['ROI'].values[0]
                plt.text(j + (i-1)*width, roi+0.01, f'{roi*100:.1f}%', 
                         ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('roi_comparison.png', dpi=300, bbox_inches='tight')
    
    # 2. Economic Outcomes in Fast vs Slow Technological Change
    plt.figure(figsize=(16, 10))
    
    # Setup subplots
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle('Economic Outcomes: Fast vs Slow Technological Change', fontsize=18)
    
    # Extract scenario data by economic scenario
    fast_tech = summary_df[summary_df['Economic'] == 'fast_tech_change']
    slow_tech = summary_df[summary_df['Economic'] == 'slow_tech_change']
    
    # Plot 1: ROI Comparison
    ax = axes[0, 0]
    x_labels = []
    roi_fast = []
    roi_slow = []
    
    for prog in programs:
        for inv in ['low', 'medium', 'high']:
            # Fast tech scenario
            fast_data = fast_tech[(fast_tech['Program'] == prog) & (fast_tech['Investment'] == inv)]
            if not fast_data.empty:
                x_labels.append(f"{prog.capitalize()}\n{inv.capitalize()} Inv.")
                roi_fast.append(fast_data['ROI'].values[0])
                
                # Corresponding slow tech
                slow_data = slow_tech[(slow_tech['Program'] == prog) & (slow_tech['Investment'] == inv)]
                if not slow_data.empty:
                    roi_slow.append(slow_data['ROI'].values[0])
                else:
                    roi_slow.append(0)
    
    x = np.arange(len(x_labels))
    width = 0.35
    
    ax.bar(x - width/2, roi_fast, width, label='Fast Tech Change', color='#FF6B6B')
    ax.bar(x + width/2, roi_slow, width, label='Slow Tech Change', color='#4ECDC4')
    
    ax.set_xlabel('Program Type and Investment Level')
    ax.set_ylabel('Return on Investment (ROI)')
    ax.set_title('ROI Comparison by Economic Scenario')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:.0f}%'))
    
    # Plot 2: Cost-Benefit Analysis
    ax = axes[0, 1]
    
    # Prepare data for grouped bar chart
    scenarios = summary_df['Scenario'].tolist()
    costs = summary_df['Cumulative_Costs'].tolist()
    benefits = summary_df['Cumulative_Benefits'].tolist()
    
    # Sort scenarios by ROI
    sorted_indices = sorted(range(len(scenarios)), key=lambda i: summary_df['ROI'].iloc[i], reverse=True)
    scenarios = [scenarios[i] for i in sorted_indices]
    costs = [costs[i] for i in sorted_indices]
    benefits = [benefits[i] for i in sorted_indices]
    
    # Truncate scenario names for display
    display_names = [s.replace(' - ', '\n').replace(' Investment', '') for s in scenarios]
    
    # Create bar positions
    x = np.arange(len(display_names))
    width = 0.35
    
    # Create bars
    ax.bar(x - width/2, costs, width, label='Costs', color='#FF6B6B')
    ax.bar(x + width/2, benefits, width, label='Benefits', color='#4ECDC4')
    
    # Add net value as text
    for i in range(len(display_names)):
        net_value = benefits[i] - costs[i]
        ax.text(i, benefits[i] + 100000, f'Net: ${net_value:,.0f}', ha='center', va='bottom', fontsize=10)
    
    # Format plot
    ax.set_xticks(x)
    ax.set_xticklabels(display_names, rotation=45, ha='right')
    ax.set_ylabel('Amount ($)')
    ax.set_title('Cost-Benefit Analysis by Scenario')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x:,.0f}'))
    
# Plot 3: Program Effectiveness Comparison
    ax = axes[1, 0]
    
    # Group by program type and economic scenario
    grouped_data = summary_df.groupby(['Program', 'Economic'])
    
    # Create bar positions
    x = np.arange(len(programs))
    width = 0.35
    
    # Collect effectiveness data
    effectiveness_fast = []
    effectiveness_slow = []
    
    for prog in programs:
        # Fast tech scenario
        fast_data = grouped_data.get_group((prog, 'fast_tech_change')) if (prog, 'fast_tech_change') in grouped_data.groups else None
        if fast_data is not None:
            effectiveness_fast.append(fast_data['Effectiveness'].mean())
        else:
            effectiveness_fast.append(0)
            
        # Slow tech scenario
        slow_data = grouped_data.get_group((prog, 'slow_tech_change')) if (prog, 'slow_tech_change') in grouped_data.groups else None
        if slow_data is not None:
            effectiveness_slow.append(slow_data['Effectiveness'].mean())
        else:
            effectiveness_slow.append(0)
    
    # Create bars
    ax.bar(x - width/2, effectiveness_fast, width, label='Fast Tech Change', color='#FF6B6B')
    ax.bar(x + width/2, effectiveness_slow, width, label='Slow Tech Change', color='#4ECDC4')
    
    # Format plot
    ax.set_xlabel('Program Type')
    ax.set_ylabel('Program Effectiveness')
    ax.set_title('Program Effectiveness by Economic Scenario')
    ax.set_xticks(x)
    ax.set_xticklabels([p.capitalize() for p in programs])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:.0f}%'))
    
    # Add value labels
    for i in range(len(programs)):
        ax.text(i - width/2, effectiveness_fast[i] + 0.01, f'{effectiveness_fast[i]*100:.1f}%', 
                ha='center', va='bottom', fontsize=10)
        ax.text(i + width/2, effectiveness_slow[i] + 0.01, f'{effectiveness_slow[i]*100:.1f}%', 
                ha='center', va='bottom', fontsize=10)
    
    # Plot 4: Skills Relevance Over Time
    ax = axes[1, 1]
    
    # Group by program type and investment level
    bubble_data = []
    
    for _, row in summary_df.iterrows():
        bubble_data.append({
            'x': row['Productivity_Gains'],
            'y': row['Skills_Relevance'],
            'size': row['ROI'] * 1000,  # Scale ROI for bubble size
            'program': row['Program'],
            'economic': row['Economic'],
            'investment': row['Investment'],
            'label': f"{row['Program'].capitalize()} - {row['Investment'].capitalize()}"
        })
    
    # Create bubble chart
    for item in bubble_data:
        color = '#FF6B6B' if item['economic'] == 'fast_tech_change' else '#4ECDC4'
        marker = 'o' if item['program'] == 'upskilling' else '^'
        ax.scatter(item['x'], item['y'], s=item['size'], color=color, alpha=0.7, marker=marker,
                  label=item['label'])
        ax.text(item['x'] + 0.005, item['y'], item['label'], fontsize=9)
    
    # Format plot
    ax.set_xlabel('Productivity Gains')
    ax.set_ylabel('Skills Relevance')
    ax.set_title('Productivity vs Skills Relevance\n(Bubble size represents ROI)')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:.0f}%'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:.0f}%'))
    
    # Remove duplicate labels
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    # No legend for this plot as we label points directly
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for suptitle
    plt.savefig('economic_outcomes_comparison.png', dpi=300, bbox_inches='tight')
    
    # 3. Investment Level Impact Analysis
    plt.figure(figsize=(16, 8))
    
    # Setup plot
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('Impact of Investment Level on Program Outcomes', fontsize=18)
    
    # Plot 1: Investment vs ROI
    ax = axes[0]
    
    # Group by program type and investment level for fast tech scenario
    fast_data = summary_df[summary_df['Economic'] == 'fast_tech_change']
    
    # Extract data for plotting
    upskilling_data = fast_data[fast_data['Program'] == 'upskilling']
    reskilling_data = fast_data[fast_data['Program'] == 'reskilling']
    
    # Map investment levels to numeric values for line plot
    investment_map = {'low': 1, 'medium': 2, 'high': 3}
    
    # Prepare upskilling data
    up_x = [investment_map[inv] for inv in upskilling_data['Investment']]
    up_y = upskilling_data['ROI'].tolist()
    up_points = sorted(zip(up_x, up_y))
    if up_points:
        up_x, up_y = zip(*up_points)
    
    # Prepare reskilling data
    re_x = [investment_map[inv] for inv in reskilling_data['Investment']]
    re_y = reskilling_data['ROI'].tolist()
    re_points = sorted(zip(re_x, re_y))
    if re_points:
        re_x, re_y = zip(*re_points)
    
    # Create line plots
    if up_points:
        ax.plot(up_x, up_y, 'o-', label='Upskilling', color='#4ECDC4', linewidth=2, markersize=10)
        for i, (x, y) in enumerate(zip(up_x, up_y)):
            ax.text(x, y + 0.01, f'{y*100:.1f}%', ha='center', va='bottom', fontsize=10)
    
    if re_points:
        ax.plot(re_x, re_y, 's-', label='Reskilling', color='#FF6B6B', linewidth=2, markersize=10)
        for i, (x, y) in enumerate(zip(re_x, re_y)):
            ax.text(x, y + 0.01, f'{y*100:.1f}%', ha='center', va='bottom', fontsize=10)
    
    # Format plot
    ax.set_xlabel('Investment Level')
    ax.set_ylabel('Return on Investment (ROI)')
    ax.set_title('ROI by Investment Level (Fast Tech Change)')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Low', 'Medium', 'High'])
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x*100:.0f}%'))
    
    # Plot 2: Cost-Benefit Ratio by Investment Level
    ax = axes[1]
    
    # Calculate cost-benefit ratio
    fast_data['CB_Ratio'] = fast_data['Cumulative_Benefits'] / fast_data['Cumulative_Costs']
    
    # Extract data for plotting
    upskilling_data = fast_data[fast_data['Program'] == 'upskilling']
    reskilling_data = fast_data[fast_data['Program'] == 'reskilling']
    
    # Prepare upskilling data
    up_x = [investment_map[inv] for inv in upskilling_data['Investment']]
    up_y = upskilling_data['CB_Ratio'].tolist()
    up_points = sorted(zip(up_x, up_y))
    if up_points:
        up_x, up_y = zip(*up_points)
    
    # Prepare reskilling data
    re_x = [investment_map[inv] for inv in reskilling_data['Investment']]
    re_y = reskilling_data['CB_Ratio'].tolist()
    re_points = sorted(zip(re_x, re_y))
    if re_points:
        re_x, re_y = zip(*re_points)
    
    # Create line plots
    if up_points:
        ax.plot(up_x, up_y, 'o-', label='Upskilling', color='#4ECDC4', linewidth=2, markersize=10)
        for i, (x, y) in enumerate(zip(up_x, up_y)):
            ax.text(x, y + 0.1, f'{y:.2f}', ha='center', va='bottom', fontsize=10)
    
    if re_points:
        ax.plot(re_x, re_y, 's-', label='Reskilling', color='#FF6B6B', linewidth=2, markersize=10)
        for i, (x, y) in enumerate(zip(re_x, re_y)):
            ax.text(x, y + 0.1, f'{y:.2f}', ha='center', va='bottom', fontsize=10)
    
    # Format plot
    ax.set_xlabel('Investment Level')
    ax.set_ylabel('Benefit-Cost Ratio')
    ax.set_title('Benefit-Cost Ratio by Investment Level (Fast Tech Change)')
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Low', 'Medium', 'High'])
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for suptitle
    plt.savefig('investment_impact_analysis.png', dpi=300, bbox_inches='tight')
    
    print("Visualizations complete and saved.")

def run_monte_carlo_analysis():
    """Run Monte Carlo simulations to analyze uncertainty in outcomes"""
    print("\nRunning Monte Carlo analysis for key scenarios...")
    
    # Define scenarios for Monte Carlo analysis
    mc_scenarios = [
        {
            'name': 'Upskilling - Fast Tech Change - High Investment',
            'config': {
                'program_type': 'upskilling',
                'economic_scenario': 'fast_tech_change',
                'investment_level': 'high',
                'company_size': 'medium',
                'simulation_years': 5
            }
        },
        {
            'name': 'Reskilling - Fast Tech Change - High Investment',
            'config': {
                'program_type': 'reskilling',
                'economic_scenario': 'fast_tech_change',
                'investment_level': 'high',
                'company_size': 'medium',
                'simulation_years': 5
            }
        }
    ]
    
    # Run Monte Carlo analysis for each scenario
    for scenario in mc_scenarios:
        print(f"Running Monte Carlo simulation for: {scenario['name']}...")
        
        # Set up the base model
        model.set_parameters(**scenario['config'])
        
        # Run Monte Carlo simulation
        mc_results = monte_carlo_simulation(model, num_simulations=100)
        
        # Plot ROI distribution
        plt.figure(figsize=(14, 8))
        plot_monte_carlo_results(mc_results, metric='ROI', confidence_interval=0.9)
        plt.title(f"Monte Carlo ROI Analysis - {scenario['name']}")
        plt.savefig(f"monte_carlo_roi_{scenario['config']['program_type']}.png", dpi=300, bbox_inches='tight')
        
        # Plot Cumulative Benefits distribution
        plt.figure(figsize=(14, 8))
        plot_monte_carlo_results(mc_results, metric='Cumulative_Benefits', confidence_interval=0.9)
        plt.title(f"Monte Carlo Benefits Analysis - {scenario['name']}")
        plt.savefig(f"monte_carlo_benefits_{scenario['config']['program_type']}.png", dpi=300, bbox_inches='tight')
        
        # Calculate risk metrics
        final_year_results = mc_results[mc_results['Year'] == model.simulation_years]
        
        # ROI risk metrics
        roi_mean = final_year_results['ROI'].mean()
        roi_std = final_year_results['ROI'].std()
        roi_min = final_year_results['ROI'].min()
        roi_max = final_year_results['ROI'].max()
        roi_median = final_year_results['ROI'].median()
        roi_5th = np.percentile(final_year_results['ROI'], 5)
        roi_95th = np.percentile(final_year_results['ROI'], 95)
        
        # Calculate probability of negative ROI
        prob_negative_roi = (final_year_results['ROI'] < 0).mean() * 100
        
        # Calculate probability of ROI > 20%
        prob_high_roi = (final_year_results['ROI'] > 0.2).mean() * 100
        
        print(f"\nMonte Carlo Analysis Results for {scenario['name']}:")
        print(f"ROI Mean: {roi_mean*100:.1f}%")
        print(f"ROI Median: {roi_median*100:.1f}%")
        print(f"ROI Standard Deviation: {roi_std*100:.1f}%")
        print(f"ROI Range: {roi_min*100:.1f}% to {roi_max*100:.1f}%")
        print(f"ROI 90% Confidence Interval: {roi_5th*100:.1f}% to {roi_95th*100:.1f}%")
        print(f"Probability of Negative ROI: {prob_negative_roi:.1f}%")
        print(f"Probability of ROI > 20%: {prob_high_roi:.1f}%")
        
    print("\nMonte Carlo analysis complete.")

def generate_strategic_recommendations():
    """Generate strategic recommendations based on simulation results"""
    print("\nGenerating strategic recommendations...")
    
    recommendations = [
        "# Strategic Recommendations for Upskilling and Reskilling Programs",
        
        "## Fast Technological Change Environment",
        
        "### For Upskilling Programs:",
        "1. **Investment Strategy**: High investment in upskilling shows diminishing returns in fast-changing environments. Maintain moderate investment levels focused on specific, adaptable skills.",
        "2. **Program Design**: Shorter, more frequent upskilling modules that can be quickly updated as technology evolves.",
        "3. **Target Areas**: Focus on foundational digital skills and adaptability rather than specific tools that may become obsolete.",
        "4. **Risk Mitigation**: Complement upskilling with selective reskilling for roles most vulnerable to technological disruption.",
        
        "### For Reskilling Programs:",
        "1. **Investment Strategy**: Higher investment levels in reskilling are justified in fast-changing technological environments, with strong ROI potential.",
        "2. **Program Design**: Comprehensive programs that build entirely new skill sets aligned with emerging technological trends.",
        "3. **Workforce Planning**: Identify roles likely to be automated or fundamentally changed and proactively reskill those employees.",
        "4. **Scalability**: Develop modular reskilling programs that can be rapidly scaled up as technological disruption accelerates.",
        
        "## Slow Technological Change Environment",
        
        "### For Upskilling Programs:",
        "1. **Investment Strategy**: Even low-investment upskilling shows strong returns in stable environments. Prioritize breadth of coverage over depth.",
        "2. **Program Design**: Longer-term upskilling initiatives that incrementally build expertise in existing domains.",
        "3. **Knowledge Retention**: Focus on institutional knowledge preservation and enhancement rather than new skill acquisition.",
        "4. **Effectiveness Metrics**: Track productivity improvements within existing roles rather than adaptation to new roles.",
        
        "### For Reskilling Programs:",
        "1. **Investment Strategy**: Be selective with reskilling investments, focusing only on critical emerging areas where skills gaps exist.",
        "2. **Program Targeting**: Reserve comprehensive reskilling for specialized roles or strategic initiatives rather than broad workforce segments.",
        "3. **ROI Timeframe**: Expect longer payback periods for reskilling investments in stable environments.",
        "4. **Hybrid Approach**: Consider blended programs that are primarily upskilling with targeted reskilling components.",
        
        "## Universal Recommendations",
        
        "1. **Economic Scenario Planning**: Regularly assess the pace of technological change in your industry and adjust the balance between upskilling and reskilling accordingly.",
        "2. **Investment Portfolio**: Maintain a balanced portfolio of both upskilling and reskilling initiatives, weighted according to technological change forecasts.",
        "3. **Measurement Framework**: Implement robust ROI and effectiveness measurement systems that account for both direct financial returns and workforce adaptability.",
        "4. **Risk Management**: Use Monte Carlo simulations to understand the range of possible outcomes before making major investments in either program type.",
        "5. **Scalability Planning**: Design programs that can be rapidly scaled up or down as economic and technological conditions change."
    ]
    
    print("\n".join(recommendations))
    return recommendations

# Main execution
if __name__ == "__main__":
    print("=" * 80)
    print("ECONOMIC SCENARIO SIMULATIONS: UPSKILLING VS RESKILLING PROGRAMS")
    print("=" * 80)
    
    # Run scenario analysis
    summary_df = run_scenario_analysis()
    
    # Create visualizations
    create_comparative_visualizations(summary_df)
    
    # Run Monte Carlo analysis
    run_monte_carlo_analysis()
    
    # Generate recommendations
    recommendations = generate_strategic_recommendations()
    
    print("\nSimulation analysis complete. All results have been saved.")